# Week 4 Practice Set – Model Evaluation & Optimization

## Objective

The objective of this practice set is to learn advanced model evaluation and optimization techniques, including Cross Validation, Hyperparameter Tuning using GridSearchCV, Model Saving with Joblib, and Loading the Saved Model for making predictions.

In [ ]:
# Import libraries

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

# Machine Learning utilities

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)

# Data preprocessing

from sklearn.preprocessing import StandardScaler

# Machine Learning model

from sklearn.svm import SVC

# Model evaluation

from sklearn.metrics import accuracy_score

# Model saving

import joblib

In [ ]:
# Load dataset

df = pd.read_csv("heart_disease_uci.csv")

# Display first five rows

df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [ ]:
# Create binary target variable

df["target"] = df["num"].apply(lambda x: 0 if x == 0 else 1)

# Display target distribution

df["target"].value_counts()

,count
target,
1,509
0,411


In [ ]:
# Fill missing values

for column in df.columns:

    if df[column].dtype in ["int64", "float64"]:
        df[column].fillna(df[column].median(), inplace=True)

    else:
        df[column].fillna(df[column].mode()[0], inplace=True)

In [ ]:
# Perform One-Hot Encoding

df = pd.get_dummies(df, drop_first=True)

# Separate input features and target variable

X = df.drop(["target", "num"], axis=1)

y = df["target"]

# Split dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Initialize StandardScaler

scaler = StandardScaler()

# Scale the data

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

**Question 1**

Use cross_val_score to test accuracy.

In [ ]:
# Create SVM model

svm_model = SVC(random_state=42)

# Perform 5-Fold Cross Validation

cv_scores = cross_val_score(
    svm_model,
    X_train_scaled,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Cross Validation Scores:", cv_scores)

print("Average Accuracy:", cv_scores.mean())

Cross Validation Scores: [0.85810811 0.86394558 0.83673469 0.79591837 0.82312925]
Average Accuracy: 0.8355671998529143


**Question 2**

Implement GridSearchCV for SVM.

In [ ]:
# Define parameter grid

param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

# Perform GridSearchCV

grid_search = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10], 'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf']},
             scoring='accuracy')

In [ ]:
# Display best parameters

print("Best Parameters:", grid_search.best_params_)

print("Best Cross Validation Score:", grid_search.best_score_)

Best Parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
Best Cross Validation Score: 0.8518477661334805


In [ ]:
# Get the best model

best_model = grid_search.best_estimator_

# Predict on test data

y_pred = best_model.predict(X_test_scaled)

# Calculate test accuracy

accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.8695652173913043


**Question 3**

Save trained model using joblib.

In [ ]:
# Save the best model

joblib.dump(best_model, "best_svm_model.pkl")

# Save the scaler

joblib.dump(scaler, "scaler.pkl")

print("Model saved successfully.")

Model saved successfully.


**Question 4**

Load model and predict on new data.

In [ ]:
# Load saved model

loaded_model = joblib.load("best_svm_model.pkl")

loaded_scaler = joblib.load("scaler.pkl")

# Select sample data

sample = X_test.iloc[:5]

# Scale sample data

sample_scaled = loaded_scaler.transform(sample)

# Predict

prediction = loaded_model.predict(sample_scaled)

print("Predictions:", prediction)

Predictions: [0 1 1 1 1]


## Observations

- Cross Validation was used to evaluate the model across multiple folds, providing a more reliable estimate of performance.
- GridSearchCV successfully identified the optimal hyperparameters for the SVM model.
- The optimized SVM model achieved improved performance compared to the default configuration.
- The trained model and scaler were saved using Joblib and successfully reloaded for making predictions on new data.

## Conclusion

This practice set demonstrated important model evaluation and optimization techniques, including Cross Validation, Hyperparameter Tuning using GridSearchCV, Model Saving, and Model Loading. These techniques improve model reliability, optimize performance, and enable trained models to be reused for future predictions.